# Kenya Youth Unemployment Analysis
## Notebook 1 — Data Cleaning & Preparation

**Author:** Abdifatah Muhlar

**Data Source:** FRED Economic Data — Federal Reserve Bank of St. Louis  
**Indicator:** SLUEM1524ZSKEN — Youth Unemployment, Kenya (Ages 15–24)  

---

### Objective
This notebook handles the initial loading, inspection, and cleaning of the
Kenya youth unemployment dataset. The goal is to produce a clean, well-documented
dataset ready for exploratory data analysis and time series modeling.

### Data Description
The dataset contains annual youth unemployment rates for Kenya from 1991 to 2025,
sourced from the World Bank via the FRED database. Youth unemployment is defined
as the percentage of the labor force aged 15–24 that is unemployed and actively
seeking employment.

In [1]:
# ============================================================
# SECTION 1 — Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 2.2.2
Numpy version: 2.0.2


---
## Section 2 — Load Dataset

The raw dataset is loaded directly from the uploaded CSV file.
We first inspect its structure before making any changes.

In [4]:
import pandas as pd
import numpy as np

# Check what files are available
import os
print(os.listdir('/content/'))

['.config', 'drive', 'sample_data']


In [5]:
import pandas as pd
import numpy as np
from google.colab import files

# This will open a file picker dialog
uploaded = files.upload()

Saving SLUEM1524ZSKEN.csv to SLUEM1524ZSKEN.csv


In [6]:
# Load the dataset
df = pd.read_csv('SLUEM1524ZSKEN.csv')

# Display basic info
print("Dataset loaded successfully")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Dataset loaded successfully
Shape: 35 rows, 2 columns

Column names: ['observation_date', 'SLUEM1524ZSKEN']

First 5 rows:


,observation_date,SLUEM1524ZSKEN
0,01/01/1991,6.209
1,01/01/1992,6.465
2,01/01/1993,6.667
3,01/01/1994,6.670
4,01/01/1995,6.513


---
## Section 3 — Data Inspection

Before cleaning, we inspect the dataset thoroughly to understand
its structure, data types, and check for any missing or anomalous values.

In [7]:
# ============================================================
# SECTION 3 — Data Inspection
# ============================================================

# Data types
print("DATA TYPES:")
print(df.dtypes)

# Missing values
print("\nMISSING VALUES:")
print(df.isnull().sum())

# Basic statistics
print("\nBASIC STATISTICS:")
print(df.describe())

# Date range
print(f"\nDate range: {df['observation_date'].iloc[0]} to {df['observation_date'].iloc[-1]}")
print(f"Total years of data: {df.shape[0]}")

DATA TYPES:
observation_date     object
SLUEM1524ZSKEN      float64
dtype: object

MISSING VALUES:
observation_date    0
SLUEM1524ZSKEN      0
dtype: int64

BASIC STATISTICS:
       SLUEM1524ZSKEN
count       35.000000
mean         8.463686
std          3.104348
min          6.209000
25%          6.660500
50%          6.885000
75%          8.215000
max         15.488000

Date range: 01/01/1991 to 01/01/2025
Total years of data: 35


---
## Section 4 — Data Cleaning

The following cleaning steps are applied:
1. Rename columns to meaningful names
2. Convert observation_date to proper datetime and extract year
3. Verify no missing values remain
4. Save cleaned dataset for use in subsequent notebooks

In [8]:
# ============================================================
# SECTION 4 — Data Cleaning
# ============================================================

# Step 1 — Rename columns
df = df.rename(columns={
    'observation_date': 'Year',
    'SLUEM1524ZSKEN': 'Unemployment_Rate'
})
print("Step 1 — Columns renamed:")
print(df.columns.tolist())

# Step 2 — Convert date and extract year
df['Year'] = pd.to_datetime(df['Year'], format='%m/%d/%Y').dt.year
print("\nStep 2 — Year extracted:")
print(df['Year'].unique())

# Step 3 — Verify no missing values
df = df.dropna()
df = df.reset_index(drop=True)
print(f"\nStep 3 — Missing values after cleaning:")
print(df.isnull().sum())

# Step 4 — Final cleaned dataset preview
print(f"\nStep 4 — Cleaned dataset ({df.shape[0]} rows):")
print(df.head(10))

Step 1 — Columns renamed:
['Year', 'Unemployment_Rate']

Step 2 — Year extracted:
[1991 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004
 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018
 2019 2020 2021 2022 2023 2024 2025]

Step 3 — Missing values after cleaning:
Year                 0
Unemployment_Rate    0
dtype: int64

Step 4 — Cleaned dataset (35 rows):
   Year  Unemployment_Rate
0  1991              6.209
1  1992              6.465
2  1993              6.667
3  1994              6.670
4  1995              6.513
5  1996              6.411
6  1997              6.536
7  1998              6.620
8  1999              6.736
9  2000              6.763


---
## Section 5 — Feature Engineering

We add additional columns that will be useful for analysis:
1. Decade — groups years into decades for comparative analysis
2. Rate_Change — year-on-year change in unemployment rate
3. Rolling_Avg — 3-year rolling average to sm

In [9]:
# ============================================================
# SECTION 5 — Feature Engineering
# ============================================================

# Decade column
df['Decade'] = (df['Year'] // 10) * 10
df['Decade'] = df['Decade'].astype(str) + 's'

# Year-on-year change
df['Rate_Change'] = df['Unemployment_Rate'].diff().round(3)

# 3-year rolling average
df['Rolling_Avg'] = df['Unemployment_Rate'].rolling(window=3).mean().round(3)

# Preview
print("Feature engineering complete:")
print(df.head(10))
print(f"\nFinal dataset shape: {df.shape}")

Feature engineering complete:
   Year  Unemployment_Rate Decade  Rate_Change  Rolling_Avg
0  1991              6.209  1990s          NaN          NaN
1  1992              6.465  1990s        0.256          NaN
2  1993              6.667  1990s        0.202        6.447
3  1994              6.670  1990s        0.003        6.601
4  1995              6.513  1990s       -0.157        6.617
5  1996              6.411  1990s       -0.102        6.531
6  1997              6.536  1990s        0.125        6.487
7  1998              6.620  1990s        0.084        6.522
8  1999              6.736  1990s        0.116        6.631
9  2000              6.763  2000s        0.027        6.706

Final dataset shape: (35, 5)


---
## Section 6 — Save Clean Dataset

The cleaned and enriched dataset is saved as a new CSV file
for use in subsequent analysis notebooks.

In [10]:
# ============================================================
# SECTION 6 — Save Clean Dataset
# ============================================================

# Save to CSV
df.to_csv('kenya_unemployment_clean.csv', index=False)

# Verify it saved
import os
if os.path.exists('kenya_unemployment_clean.csv'):
    print("Clean dataset saved successfully")
    print(f"File size: {os.path.getsize('kenya_unemployment_clean.csv')} bytes")
else:
    print("ERROR — file not saved")

# Final summary
print(f"""
CLEANING SUMMARY
================
Original columns    : observation_date, SLUEM1524ZSKEN
Cleaned columns     : Year, Unemployment_Rate, Decade, Rate_Change, Rolling_Avg
Original rows       : 35
Cleaned rows        : 35
Missing values      : 0
Year range          : 1991 — 2025
Output file         : kenya_unemployment_clean.csv
""")

Clean dataset saved successfully
File size: 1072 bytes

CLEANING SUMMARY
Original columns    : observation_date, SLUEM1524ZSKEN
Cleaned columns     : Year, Unemployment_Rate, Decade, Rate_Change, Rolling_Avg
Original rows       : 35
Cleaned rows        : 35
Missing values      : 0
Year range          : 1991 — 2025
Output file         : kenya_unemployment_clean.csv



In [11]:
from google.colab import files

files.download('kenya_unemployment_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>